In [1]:
import os
import json
import warnings
import logging
from pathlib import Path
from typing import Optional, List, Tuple

In [2]:
import numpy as np
import pandas as pd
import optuna
import matplotlib.pyplot as plt
import matplotlib
matplotlib.use("Agg")                        # headless rendering in Kaggle
from catboost import CatBoostClassifier, Pool
from sklearn.metrics import log_loss
 
warnings.filterwarnings("ignore")
optuna.logging.set_verbosity(optuna.logging.WARNING)
logging.basicConfig(level=logging.INFO, format="%(asctime)s  %(message)s")
log = logging.getLogger(__name__)

In [3]:
# ══════════════════════════════════════════════════════════════════════════
# CONFIGURATION
# ══════════════════════════════════════════════════════════════════════════
DATA_DIR   = Path("/kaggle/input/competitions/march-machine-learning-mania-2026")
OUTPUT_DIR = Path("/kaggle/working")
 
# ELO hyper-parameters
ELO_K               = 30          # base K-factor
ELO_BASE            = 1500        # starting / mean ELO
ELO_REGRESS_FRAC    = 0.40        # fraction to regress toward mean each season
ELO_MOV_MULTIPLIER  = 0.006       # margin-of-victory dampening constant
HOME_ADVANTAGE      = 100         # ELO points for home-court edge
 
# Optuna
N_OPTUNA_TRIALS = 60
OPTUNA_TIMEOUT  = 3600           # seconds
 
# Validation split
TRAIN_SEASONS_MAX = 2024
VAL_SEASON        = 2025
PREDICT_SEASON    = 2026          # inferred from sample submission
 
# Seed for reproducibility
SEED = 42

In [4]:
# ══════════════════════════════════════════════════════════════════════════
# 1.  DATA LOADING
# ══════════════════════════════════════════════════════════════════════════
 
def load_data(gender: str = "M") -> dict:
    """
    Load all relevant CSV files for the given gender ('M' or 'W').
    Returns a dict keyed by short name.
    """
    g = gender  # 'M' or 'W'
    files = {
        "reg_compact":    f"{g}RegularSeasonCompactResults.csv",
        "reg_detailed":   f"{g}RegularSeasonDetailedResults.csv",
        "tourney_compact":f"{g}NCAATourneyCompactResults.csv",
        "tourney_seeds":  f"{g}NCAATourneySeeds.csv",
        "teams":          f"{g}Teams.csv",
        "seasons":        f"{g}Seasons.csv",
        "conferences":    f"{g}TeamConferences.csv",
    }
    # Men-only files
    if g == "M":
        files["massey"] = "MMasseyOrdinals.csv"
 
    data = {}
    for key, fname in files.items():
        fpath = DATA_DIR / fname
        if fpath.exists():
            data[key] = pd.read_csv(fpath)
            log.info(f"  Loaded {fname:50s}  shape={data[key].shape}")
        else:
            log.warning(f"  File not found: {fname}")
            data[key] = pd.DataFrame()
 
    # Prefer Stage2 sample submission (final scoring); fall back to Stage1
    for _sf in ["SampleSubmissionStage2.csv", "SampleSubmissionStage1.csv"]:
        _sp = DATA_DIR / _sf
        if _sp.exists():
            data["sample_sub"] = pd.read_csv(_sp)
            log.info(f"  Loaded {_sf}  shape={data['sample_sub'].shape}")
            break
    else:
        raise FileNotFoundError("No SampleSubmission CSV found in DATA_DIR")
    return data

In [5]:
# ══════════════════════════════════════════════════════════════════════════
# 2.  ELO RATING ENGINE
# ══════════════════════════════════════════════════════════════════════════
 
def expected_score(elo_a: float, elo_b: float) -> float:
    """Standard ELO expected score for team A."""
    return 1.0 / (1.0 + 10 ** ((elo_b - elo_a) / 400.0))
 
 
def mov_multiplier(score_diff: float, elo_diff: float) -> float:
    """
    Margin-of-Victory multiplier used by FiveThirtyEight-style ELO.
    Dampens large blowouts so they don't wildly shift ratings.
    """
    return np.log(abs(score_diff) + 1) * (2.2 / (elo_diff * ELO_MOV_MULTIPLIER + 2.2))
 
 
def compute_elo_ratings(reg_compact: pd.DataFrame) -> pd.DataFrame:
    """
    Compute season-end ELO ratings for every team from regular-season games.
 
    Processing order: Season (asc) → DayNum (asc)
    Season regression: at start of each new season regress 40 % toward mean.
 
    Returns DataFrame with columns: Season, TeamID, ELO
    """
    log.info("Computing ELO ratings …")
 
    # Sort chronologically
    games = reg_compact.sort_values(["Season", "DayNum"]).copy()
 
    elo_ratings: dict[int, float] = {}   # TeamID → current ELO
    records = []                          # (Season, TeamID, ELO) snapshots
 
    current_season = None
 
    for row in games.itertuples(index=False):
        season = int(row.Season)
        wteam  = int(row.WTeamID)
        lteam  = int(row.LTeamID)
        wscore = int(row.WScore)
        lscore = int(row.LScore)
    
        loc = getattr(row, "WLoc", "N")
 
        # ── Season boundary: regress toward mean ──────────────────────────
        if season != current_season:
            if current_season is not None:
                # Save end-of-previous-season ELO
                for tid, elo in elo_ratings.items():
                    records.append({"Season": current_season, "TeamID": tid, "ELO": elo})
            current_season = season
            # Regress all existing ratings toward ELO_BASE
            elo_ratings = {
                tid: ELO_BASE + (1 - ELO_REGRESS_FRAC) * (elo - ELO_BASE)
                for tid, elo in elo_ratings.items()
            }
            # Initialise teams that have never appeared
            for tid in [wteam, lteam]:
                if tid not in elo_ratings:
                    elo_ratings[tid] = ELO_BASE
 
        # Ensure both teams are initialised
        for tid in [wteam, lteam]:
            if tid not in elo_ratings:
                elo_ratings[tid] = ELO_BASE
 
        elo_w = elo_ratings[wteam]
        elo_l = elo_ratings[lteam]
 
        # Home-court adjustment for expected score
        if loc == "H":
            exp_w = expected_score(elo_w + HOME_ADVANTAGE, elo_l)
        elif loc == "A":
            exp_w = expected_score(elo_w, elo_l + HOME_ADVANTAGE)
        else:
            exp_w = expected_score(elo_w, elo_l)
 
        # Margin of victory multiplier
        score_diff = wscore - lscore
        elo_diff   = elo_w - elo_l
        k_adjusted = ELO_K * mov_multiplier(score_diff, elo_diff)
 
        # FIX: zero-sum update — loser loses k*exp_w, not k*(1-exp_w)
        elo_ratings[wteam] = elo_w + k_adjusted * (1 - exp_w)
        elo_ratings[lteam] = elo_l - k_adjusted * exp_w
 
    # Save final season
    if current_season is not None:
        for tid, elo in elo_ratings.items():
            records.append({"Season": current_season, "TeamID": tid, "ELO": elo})
 
    elo_df = pd.DataFrame(records)
    log.info(f"ELO ratings computed: {len(elo_df)} rows, {elo_df['Season'].nunique()} seasons")
    return elo_df

In [6]:
# ══════════════════════════════════════════════════════════════════════════
# 3.  TEAM FEATURES  (regular-season stats only — no leakage)
# ══════════════════════════════════════════════════════════════════════════
 
def build_team_features(
    reg_compact: pd.DataFrame,
    tourney_seeds: pd.DataFrame,
    elo_df: pd.DataFrame,
    massey: Optional[pd.DataFrame] = None,
) -> pd.DataFrame:
    """
    Build one row per (Season, TeamID) with:
        WinPct, AvgPtsFor, AvgPtsAgainst, AvgMargin,
        Seed (numeric), ELO, optionally MasseyRank
    """
    log.info("Building team features …")
 
    # ── Win/loss stats ─────────────────────────────────────────────────────
    wins = reg_compact.rename(columns={"WTeamID": "TeamID", "WScore": "PtsFor", "LScore": "PtsAgainst"})
    wins = wins[["Season", "TeamID", "PtsFor", "PtsAgainst"]].copy()
    wins["Win"] = 1
 
    losses = reg_compact.rename(columns={"LTeamID": "TeamID", "LScore": "PtsFor", "WScore": "PtsAgainst"})
    losses = losses[["Season", "TeamID", "PtsFor", "PtsAgainst"]].copy()
    losses["Win"] = 0
 
    all_games = pd.concat([wins, losses], ignore_index=True)
    all_games["Margin"] = all_games["PtsFor"] - all_games["PtsAgainst"]
 
    stats = (
        all_games.groupby(["Season", "TeamID"])
        .agg(
            Games       = ("Win",       "count"),
            Wins        = ("Win",       "sum"),
            AvgPtsFor   = ("PtsFor",    "mean"),
            AvgPtsAgainst=("PtsAgainst","mean"),
            AvgMargin   = ("Margin",    "mean"),
        )
        .reset_index()
    )
    stats["WinPct"] = stats["Wins"] / stats["Games"]
 
    # ── Tournament seeds ───────────────────────────────────────────────────
    seeds = tourney_seeds.copy()
    # Seed format: 'W01', 'X16a' → extract integer
    seeds["SeedNum"] = (
        seeds["Seed"]
        .str.extract(r"(\d+)", expand=False)
        .astype(float)
    )
    seeds = seeds[["Season", "TeamID", "SeedNum"]].drop_duplicates()
 
    # ── Massey Ordinals (latest ranking snapshot pre-tournament) ───────────
    if massey is not None and not massey.empty:
        # Keep only rankings available before tournament (DayNum ≤ 133)
        m = massey[massey["RankingDayNum"] <= 133].copy()
        # Average rank across all ranking systems, lower = better
        m_avg = (
            m.groupby(["Season", "TeamID"])["OrdinalRank"]
            .mean()
            .reset_index()
            .rename(columns={"OrdinalRank": "AvgMasseyRank"})
        )
    else:
        m_avg = pd.DataFrame(columns=["Season", "TeamID", "AvgMasseyRank"])
 
    # ── Merge everything ───────────────────────────────────────────────────
    team_features = (
        stats
        .merge(seeds, on=["Season", "TeamID"], how="left")
        .merge(elo_df, on=["Season", "TeamID"], how="left")
        .merge(m_avg,  on=["Season", "TeamID"], how="left")
    )
 
    # Fill missing seeds (teams not in tournament) with 16 (worst seed)
    team_features["SeedNum"] = team_features["SeedNum"].fillna(16.0)
 
    # Fill missing Massey with a bad rank (999)
    team_features["AvgMasseyRank"] = team_features["AvgMasseyRank"].fillna(999.0)
 
    # Fill missing ELO with base
    team_features["ELO"] = team_features["ELO"].fillna(ELO_BASE)
 
    log.info(f"Team features shape: {team_features.shape}")
    return team_features

In [7]:
# ══════════════════════════════════════════════════════════════════════════
# 4.  TRAINING DATA CONSTRUCTION
# ══════════════════════════════════════════════════════════════════════════
 
FEATURE_COLS = [
    "TeamA_ELO", "TeamB_ELO", "ELO_diff",
    "TeamA_WinPct", "TeamB_WinPct", "WinPct_diff",
    "TeamA_AvgPtsFor", "TeamB_AvgPtsFor",
    "TeamA_AvgPtsAgainst", "TeamB_AvgPtsAgainst",
    "TeamA_AvgMargin", "TeamB_AvgMargin", "Margin_diff",
    "TeamA_SeedNum", "TeamB_SeedNum", "Seed_diff",
    "TeamA_AvgMasseyRank", "TeamB_AvgMasseyRank",
]
 
 
def _attach_features(df: pd.DataFrame, team_features: pd.DataFrame) -> pd.DataFrame:
    """
    Merge team_features for TeamA and TeamB into a matchup DataFrame.
    df must have columns: Season, TeamA, TeamB
    """
    tf = team_features.rename(columns=lambda c: f"TeamA_{c}" if c not in ["Season","TeamID"] else c)
    df = df.merge(tf.rename(columns={"TeamID": "TeamA"}), on=["Season","TeamA"], how="left")
 
    tf = team_features.rename(columns=lambda c: f"TeamB_{c}" if c not in ["Season","TeamID"] else c)
    df = df.merge(tf.rename(columns={"TeamID": "TeamB"}), on=["Season","TeamB"], how="left")
 
    # Difference features
    df["ELO_diff"]     = df["TeamA_ELO"]     - df["TeamB_ELO"]
    df["WinPct_diff"]  = df["TeamA_WinPct"]  - df["TeamB_WinPct"]
    df["Margin_diff"]  = df["TeamA_AvgMargin"]- df["TeamB_AvgMargin"]
    df["Seed_diff"]    = df["TeamA_SeedNum"]  - df["TeamB_SeedNum"]
    return df
 
 
def create_training_data(
    tourney_compact: pd.DataFrame,
    team_features: pd.DataFrame,
) -> pd.DataFrame:
    """
    Build one row per historical tournament game with:
      - TeamA = lower TeamID, TeamB = higher TeamID
      - Result = 1 if TeamA won
      - All engineered features
    """
    log.info("Creating training data …")
 
    df = tourney_compact.copy()
 
    # Standardise: TeamA < TeamB
    df["TeamA"] = df[["WTeamID","LTeamID"]].min(axis=1)
    df["TeamB"] = df[["WTeamID","LTeamID"]].max(axis=1)
    df["Result"] = (df["WTeamID"] == df["TeamA"]).astype(int)
 
    df = df[["Season","TeamA","TeamB","Result"]].copy()
    df = _attach_features(df, team_features)
 
    # Drop rows where core features are entirely missing
    df = df.dropna(subset=["TeamA_ELO","TeamB_ELO"])
 
    # Keep only defined feature cols + meta
    avail = [c for c in FEATURE_COLS if c in df.columns]
    out = df[["Season","TeamA","TeamB","Result"] + avail].copy()
 
    log.info(f"Training data shape: {out.shape} | seasons {sorted(out['Season'].unique())}")
    return out

In [8]:
# ══════════════════════════════════════════════════════════════════════════
# 5.  CATBOOST + OPTUNA
# ══════════════════════════════════════════════════════════════════════════
 
def train_catboost_with_optuna(
    train_df: pd.DataFrame,
    val_df: pd.DataFrame,
    feature_cols: list[str],
    n_trials: int = N_OPTUNA_TRIALS,
) -> Tuple[CatBoostClassifier, optuna.Study]:
    """
    Tune CatBoostClassifier with Optuna, then refit best params on
    train + val combined.
 
    Returns (final_model, study).
    """
    X_train = train_df[feature_cols].values
    y_train = train_df["Result"].values
    X_val   = val_df[feature_cols].values
    y_val   = val_df["Result"].values
 
    train_pool = Pool(X_train, label=y_train)
    val_pool   = Pool(X_val,   label=y_val)
 
    # ── Optuna objective ───────────────────────────────────────────────────
    def objective(trial: optuna.Trial) -> float:
        params = {
            "iterations":          trial.suggest_int("iterations", 300, 2000),
            "depth":               trial.suggest_int("depth", 4, 10),
            "learning_rate":       trial.suggest_float("learning_rate", 0.01, 0.3, log=True),
            "l2_leaf_reg":         trial.suggest_float("l2_leaf_reg", 1.0, 10.0),
            "random_strength":     trial.suggest_float("random_strength", 0.1, 10.0),
            "bagging_temperature": trial.suggest_float("bagging_temperature", 0.0, 1.0),
            "border_count":        trial.suggest_int("border_count", 32, 255),
            "loss_function":       "Logloss",
            "eval_metric":         "Logloss",
            "random_seed":         SEED,
            "verbose":             False,
        }
        # FIX: early_stopping_rounds / use_best_model are .fit() args, not constructor args
        model = CatBoostClassifier(**params)
        model.fit(
            train_pool,
            eval_set=val_pool,
            verbose=False,
            early_stopping_rounds=50,
            use_best_model=True,
        )
        preds = model.predict_proba(X_val)[:, 1]
        return log_loss(y_val, preds)
 
    log.info(f"Running Optuna ({n_trials} trials) …")
    study = optuna.create_study(
        direction="minimize",
        sampler=optuna.samplers.TPESampler(seed=SEED),
    )
    study.optimize(objective, n_trials=n_trials, timeout=OPTUNA_TIMEOUT, show_progress_bar=False)
 
    best_params = study.best_params
    best_val_loss = study.best_value
    log.info(f"Best val log-loss = {best_val_loss:.5f}")
    log.info(f"Best params: {best_params}")
 
    # ── Retrain on train + val with best params ────────────────────────────
    X_all = np.vstack([X_train, X_val])
    y_all = np.concatenate([y_train, y_val])
 
    final_params = {
        **best_params,
        "loss_function":  "Logloss",
        "eval_metric":    "Logloss",
        "random_seed":    SEED,
        "verbose":        100,
    }
    final_model = CatBoostClassifier(**final_params)
    final_model.fit(Pool(X_all, label=y_all))
 
    return final_model, study

In [9]:
# ══════════════════════════════════════════════════════════════════════════
# 6.  FEATURE IMPORTANCE PLOT
# ══════════════════════════════════════════════════════════════════════════
 
def plot_feature_importance(model: CatBoostClassifier, feature_cols: list[str]) -> None:
    """Save a bar chart of CatBoost feature importances."""
    importances = model.get_feature_importance()
    fi = pd.Series(importances, index=feature_cols).sort_values(ascending=True)
 
    fig, ax = plt.subplots(figsize=(9, max(4, len(fi) * 0.35)))
    fi.plot(kind="barh", ax=ax, color="steelblue")
    ax.set_title("CatBoost Feature Importances", fontsize=13)
    ax.set_xlabel("Importance")
    plt.tight_layout()
    out_path = OUTPUT_DIR / "feature_importance.png"
    plt.savefig(out_path, dpi=120)
    plt.close()
    log.info(f"Feature importance chart saved → {out_path}")

In [10]:
# ══════════════════════════════════════════════════════════════════════════
# 7.  SUBMISSION GENERATION
# ══════════════════════════════════════════════════════════════════════════
 
def build_submission_features(
    sample_sub: pd.DataFrame,
    team_features: pd.DataFrame,
) -> pd.DataFrame:
    """
    Parse SampleSubmissionStage1.csv, attach features, return ready DataFrame.
    """
    log.info("Building submission features …")
 
    # Parse ID column: Season_TeamA_TeamB
    sub = sample_sub.copy()
    parts   = sub["ID"].str.split("_", expand=True)
    sub["Season"] = parts[0].astype(int)
    sub["TeamA"]  = parts[1].astype(int)
    sub["TeamB"]  = parts[2].astype(int)
 
    sub = _attach_features(sub, team_features)
    log.info(f"Submission features shape: {sub.shape}")
    return sub
 
 
def generate_submission(
    model: CatBoostClassifier,
    sub_df: pd.DataFrame,
    feature_cols: list[str],
    out_path: Path = OUTPUT_DIR / "submission.csv",
) -> pd.DataFrame:
    """
    Generate clipped predictions and save submission.csv.
    """
    avail = [c for c in feature_cols if c in sub_df.columns]
    X_sub = sub_df[avail].fillna(ELO_BASE).values   # safety fill
 
    preds = model.predict_proba(X_sub)[:, 1]
    preds = np.clip(preds, 0.001, 0.999)
 
    submission = pd.DataFrame({"ID": sub_df["ID"], "Pred": preds})
    submission.to_csv(out_path, index=False)
    log.info(f"Submission saved → {out_path}  ({len(submission)} rows)")
    log.info(f"  Pred stats: min={preds.min():.4f}  max={preds.max():.4f}  mean={preds.mean():.4f}")
    return submission

In [11]:
# ══════════════════════════════════════════════════════════════════════════
# 8.  PIPELINE HELPERS  (men + women combined)
# ══════════════════════════════════════════════════════════════════════════
 
def run_pipeline(gender: str) -> Tuple[pd.DataFrame, CatBoostClassifier, List[str], pd.DataFrame]:
    """
    Full pipeline for a single gender.
    Returns (team_features_latest_season, model, feature_cols).
    """
    log.info(f"\n{'='*60}")
    log.info(f"  RUNNING PIPELINE  gender={gender}")
    log.info(f"{'='*60}")
 
    # 1. Load
    data = load_data(gender)
 
    reg_compact     = data["reg_compact"]
    tourney_compact = data["tourney_compact"]
    tourney_seeds   = data["tourney_seeds"]
    massey          = data.get("massey")
 
    # 2. ELO
    elo_df = compute_elo_ratings(reg_compact)
 
    # 3. Team features
    team_features = build_team_features(
        reg_compact, tourney_seeds, elo_df,
        massey if (massey is not None and not massey.empty) else None,
    )
 
    # 4. Training data
    train_data = create_training_data(tourney_compact, team_features)
 
    # Decide feature cols (only those present)
    feature_cols = [c for c in FEATURE_COLS if c in train_data.columns]
    log.info(f"Using {len(feature_cols)} features: {feature_cols}")
 
    # 5. Train / val split
    train_df = train_data[train_data["Season"] <= TRAIN_SEASONS_MAX].copy()
    val_df   = train_data[train_data["Season"] == VAL_SEASON].copy()
 
    log.info(f"Train rows: {len(train_df)}  |  Val rows: {len(val_df)}")
 
    if val_df.empty:
        log.warning(f"No validation data for season {VAL_SEASON} — using last 2 seasons as val.")
        val_df   = train_data[train_data["Season"] >= train_data["Season"].max() - 1].copy()
        train_df = train_data[~train_data.index.isin(val_df.index)].copy()
 
    # 6. Optuna + CatBoost
    model, study = train_catboost_with_optuna(train_df, val_df, feature_cols)


    model_path = OUTPUT_DIR / f"{gender}_catboost_model.cbm"
    model.save_model(model_path)
    
    params_path = OUTPUT_DIR / f"{gender}_best_params.json"
    with open(params_path, "w") as f:
        json.dump(study.best_params, f)

    log.info(f"Saved model → {model_path}")
    log.info(f"Saved params → {params_path}")
    
    # 7. Feature importance
    plot_feature_importance(model, feature_cols)
 
    # 8. Return team features for prediction season (latest)
    max_season_in_features = team_features["Season"].max()
    log.info(f"Latest season in team_features: {max_season_in_features}")
    # For prediction we need features for the 2026 teams.
    # If 2026 regular season data is available, use it; otherwise fall back to latest.
    pred_season = PREDICT_SEASON
    if pred_season not in team_features["Season"].values:
        log.warning(
            f"Season {pred_season} not found in team_features; "
            f"using {max_season_in_features} instead."
        )
        pred_season = max_season_in_features
 
    team_features_pred = team_features[team_features["Season"] == pred_season].copy()
    # Override Season label so merge keys match the submission IDs
    team_features_pred["Season"] = PREDICT_SEASON
 
    return team_features_pred, model, feature_cols, data["sample_sub"]

In [12]:
# ══════════════════════════════════════════════════════════════════════════
# 9.  MAIN
# ══════════════════════════════════════════════════════════════════════════
 
def main() -> None:
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
 
    all_submissions = []
 
    # ── Determine which genders appear in the sample submission ───────────
    # Prefer Stage2 sample submission (final scoring); fall back to Stage1
    for _sf in ["SampleSubmissionStage2.csv", "SampleSubmissionStage1.csv"]:
        _sp = DATA_DIR / _sf
        if _sp.exists():
            sample_sub_raw = pd.read_csv(_sp)
            log.info(f"Loaded {_sf} ({len(sample_sub_raw)} rows)")
            break
    else:
        raise FileNotFoundError("No SampleSubmission CSV found in DATA_DIR")
    # Infer gender by checking team IDs ranges or by running both and filtering
    # We run both pipelines and combine predictions.
 
    for gender in ["M", "W"]:
        reg_path = DATA_DIR / f"{gender}RegularSeasonCompactResults.csv"
        if not reg_path.exists():
            log.warning(f"No data found for gender {gender}, skipping.")
            continue
 
        try:
            team_features_pred, model, feature_cols, sample_sub = run_pipeline(gender)
 
            # Filter sample submission to this gender's teams
            # Men's teams are typically < 2000 (or in MTeams); women's ≥ 3000
            sub_df = build_submission_features(sample_sub, team_features_pred)
 
            # Keep only rows that actually belong to this gender by checking
            # whether TeamA exists in team_features_pred
            valid_teams = set(team_features_pred["TeamID"].unique())
            mask = sub_df["TeamA"].isin(valid_teams) & sub_df["TeamB"].isin(valid_teams)
            sub_df_filtered = sub_df[mask].copy()
 
            if sub_df_filtered.empty:
                log.warning(f"No matching submission rows for gender {gender}. Skipping.")
                continue
 
            avail = [c for c in feature_cols if c in sub_df_filtered.columns]
            X_sub = sub_df_filtered[avail].fillna(ELO_BASE).values
            preds = np.clip(model.predict_proba(X_sub)[:, 1], 0.001, 0.999)
 
            sub_df_filtered = sub_df_filtered.copy()
            sub_df_filtered["Pred"] = preds
            all_submissions.append(sub_df_filtered[["ID","Pred"]])
            log.info(f"Gender {gender}: {len(sub_df_filtered)} predictions generated.")
 
        except Exception as exc:
            log.error(f"Pipeline failed for gender {gender}: {exc}", exc_info=True)
 
    # ── Combine and fill any missing rows with 0.5 ─────────────────────────
    if not all_submissions:
        raise RuntimeError("No predictions generated! Check data paths.")
 
    combined = pd.concat(all_submissions, ignore_index=True)
 
    # Merge with full sample submission to ensure all rows are covered
    final_sub = sample_sub_raw[["ID"]].merge(combined, on="ID", how="left")
    final_sub["Pred"] = final_sub["Pred"].fillna(0.5)
 
    out_path = OUTPUT_DIR / "submission.csv"
    final_sub.to_csv(out_path, index=False)
    log.info(f"\nFinal submission: {out_path}  ({len(final_sub)} rows)")
    log.info(f"   Pred stats: min={final_sub['Pred'].min():.4f}  "
             f"max={final_sub['Pred'].max():.4f}  "
             f"mean={final_sub['Pred'].mean():.4f}")
 
    # Preview
    print("\n=== Submission Preview ===")
    print(final_sub.head(10).to_string(index=False))
 
 
# ══════════════════════════════════════════════════════════════════════════
 
if __name__ == "__main__":
    main()

2026-04-05 06:23:35,830  Loaded SampleSubmissionStage2.csv (132133 rows)
2026-04-05 06:23:35,832  
2026-04-05 06:23:35,832    RUNNING PIPELINE  gender=M
2026-04-05 06:23:35,833  ============================================================
2026-04-05 06:23:35,995    Loaded MRegularSeasonCompactResults.csv                    shape=(198577, 8)
2026-04-05 06:23:36,399    Loaded MRegularSeasonDetailedResults.csv                   shape=(124529, 34)
2026-04-05 06:23:36,411    Loaded MNCAATourneyCompactResults.csv                      shape=(2585, 8)
2026-04-05 06:23:36,420    Loaded MNCAATourneySeeds.csv                               shape=(2694, 3)
2026-04-05 06:23:36,427    Loaded MTeams.csv                                          shape=(381, 4)
2026-04-05 06:23:36,436    Loaded MSeasons.csv                                        shape=(42, 6)
2026-04-05 06:23:36,449    Loaded MTeamConferences.csv                                shape=(13753, 3)
2026-04-05 06:23:39,786    Loaded MMasseyOrd

0:	learn: 0.6536268	total: 9.21ms	remaining: 16.6s
100:	learn: 0.1331708	total: 693ms	remaining: 11.7s
200:	learn: 0.0428267	total: 1.34s	remaining: 10.7s
300:	learn: 0.0209068	total: 1.99s	remaining: 9.94s
400:	learn: 0.0126728	total: 2.63s	remaining: 9.18s
500:	learn: 0.0088254	total: 3.25s	remaining: 8.44s
600:	learn: 0.0068201	total: 3.88s	remaining: 7.76s
700:	learn: 0.0058767	total: 4.5s	remaining: 7.07s
800:	learn: 0.0052520	total: 5.13s	remaining: 6.42s
900:	learn: 0.0049103	total: 5.77s	remaining: 5.78s
1000:	learn: 0.0047787	total: 6.41s	remaining: 5.14s
1100:	learn: 0.0046456	total: 7.04s	remaining: 4.49s
1200:	learn: 0.0045062	total: 7.7s	remaining: 3.86s
1300:	learn: 0.0044689	total: 8.32s	remaining: 3.21s
1400:	learn: 0.0044013	total: 8.96s	remaining: 2.57s
1500:	learn: 0.0043776	total: 9.62s	remaining: 1.94s
1600:	learn: 0.0043031	total: 10.3s	remaining: 1.3s
1700:	learn: 0.0042878	total: 10.9s	remaining: 656ms
1800:	learn: 0.0042722	total: 11.5s	remaining: 12.8ms
1802:	

2026-04-05 06:24:43,089  Saved model → /kaggle/working/M_catboost_model.cbm
2026-04-05 06:24:43,090  Saved params → /kaggle/working/M_best_params.json
2026-04-05 06:24:43,632  Feature importance chart saved → /kaggle/working/feature_importance.png
2026-04-05 06:24:43,633  Latest season in team_features: 2026
2026-04-05 06:24:43,655  Building submission features …
2026-04-05 06:24:44,181  Submission features shape: (132133, 27)
2026-04-05 06:24:44,448  Gender M: 66430 predictions generated.
2026-04-05 06:24:44,451  
2026-04-05 06:24:44,452    RUNNING PIPELINE  gender=W
2026-04-05 06:24:44,452  ============================================================
2026-04-05 06:24:44,551    Loaded WRegularSeasonCompactResults.csv                    shape=(142507, 8)
2026-04-05 06:24:44,829    Loaded WRegularSeasonDetailedResults.csv                   shape=(87187, 34)
2026-04-05 06:24:44,838    Loaded WNCAATourneyCompactResults.csv                      shape=(1717, 8)
2026-04-05 06:24:44,846    Lo

0:	learn: 0.6044603	total: 1.49ms	remaining: 640ms
100:	learn: 0.2081567	total: 141ms	remaining: 457ms
200:	learn: 0.1083067	total: 269ms	remaining: 306ms
300:	learn: 0.0614658	total: 398ms	remaining: 169ms


2026-04-05 06:25:31,713  Saved model → /kaggle/working/W_catboost_model.cbm
2026-04-05 06:25:31,714  Saved params → /kaggle/working/W_best_params.json


400:	learn: 0.0379513	total: 525ms	remaining: 36.6ms
428:	learn: 0.0341926	total: 562ms	remaining: 0us


2026-04-05 06:25:32,031  Feature importance chart saved → /kaggle/working/feature_importance.png
2026-04-05 06:25:32,032  Latest season in team_features: 2026
2026-04-05 06:25:32,040  Building submission features …
2026-04-05 06:25:32,441  Submission features shape: (132133, 27)
2026-04-05 06:25:32,515  Gender W: 65703 predictions generated.
2026-04-05 06:25:32,873  
Final submission: /kaggle/working/submission.csv  (132133 rows)
2026-04-05 06:25:32,875     Pred stats: min=0.0010  max=0.9990  mean=0.4904



=== Submission Preview ===
            ID     Pred
2026_1101_1102 0.912598
2026_1101_1103 0.044021
2026_1101_1104 0.002325
2026_1101_1105 0.260166
2026_1101_1106 0.813741
2026_1101_1107 0.574672
2026_1101_1108 0.984365
2026_1101_1110 0.920649
2026_1101_1111 0.195837
2026_1101_1112 0.001000
